In [0]:
## Spark Performance Settings
spark.conf.set("spark.sql.shuffle.partitions", "auto")
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")

In [0]:
def get_widget_value(name, default):
    try:
        dbutils.widgets.text(name, str(default))
        value = dbutils.widgets.get(name)
        return value if value not in (None, "") else default
    except NameError:
        return default
    
def validate_positive_int(name, value):
    parsed_value = int(value)
    if parsed_value <= 0:
        raise ValueError(f"Invalid widget value for {name}: {value}. Value must be a positive integer.")
    return parsed_value



In [0]:
training_run = get_widget_value("training_run", "True")
start_date = get_widget_value("start_date", "2023-01-01")
catalog_schema_prefix = get_widget_value("catalog_schema_prefix", "marketingdata_dev.claire_wilsonbarnes")
ranking_validation_run =get_widget_value("ranking_validation_run", "False")

In [0]:
%
# dbutils.widgets.text(name="start_date", defaultValue="2026-05-07", label="start_date")
# dbutils.widgets.text(name="lookback_period", defaultValue="30", label="lookback_period")
# dbutils.widgets.text(name="year_lookback_period", defaultValue="365", label="year_lookback_period")
# dbutils.widgets.text(name="catalog_schema_prefix", defaultValue="marketingdata_dev.claire_wilsonbarnes", label="catalog_schema_prefix")
# dbutils.widgets.text(name="training_run", defaultValue="True", label="training_run")
# dbutils.widgets.text(name="earliest_date", defaultValue="2025-12-01", label="earliest_date")
# dbutils.widgets.text(name="latest_date", defaultValue="2026-05-27", label="latest_date")

In [0]:
%sql
-- Table of all run dates 
CREATE OR REPLACE TABLE IDENTIFIER(:catalog_schema_prefix || '.pctr_training_dates') AS (
SELECT CURRENT_DATE - interval 1 day AS rundate
);

In [0]:
%sql
CREATE OR REPLACE TABLE IDENTIFIER(:catalog_schema_prefix || '.pctr_training_ads_base') AS (
WITH cte_all_ads AS (
SELECT
      c.UniqueAdID
    , c.rundate
    , c.PotNumber
    , UPPER(c.CampaignNumber) AS campaignnumber
    , REGEXP_EXTRACT(c.UniqueAdID, '^.*_(V[1-9])_.*$', 1) AS versionnumber
    , c.Title 
    , c.AlgoDivision
    , c.TradeDivision
    , c.Items 
    , regexp_replace(c.Themes, '[^a-zA-Z0-9]', '')  AS theme
    -- Can possibly improve on this with more logic added for this but is a starting point!
    , CASE WHEN LOWER(c.UniqueAdID) LIKE ANY('%fathers%', '%mothers%', '%christmas%', '%easter%', '%valentine%', '%halloween%', '%eid%') THEN 1 ELSE 0 END AS seasonal_flag
  FROM
    IDENTIFIER(:catalog_schema_prefix || '.pctr_training_dates') AS d 
    INNER JOIN marketingdata_prod.warehouse.next_uk_nextads_control_sheet_latest AS c
      ON d.rundate=c.rundate
  WHERE 
    -- FILTERED ATM FOR SHOPPINGBAG- if to expand will need to build this out as part of the additional data columns 
     c.PageGroup ='ShoppingBag' 
  -- Group due to 2 diff SB locations 
  GROUP BY 
      c.UniqueAdID
    , c.rundate
    , c.PotNumber
    , UPPER(c.CampaignNumber)
    , REGEXP_EXTRACT(c.UniqueAdID, '^.*_(V[1-9])_.*$', 1) 
    , c.Title 
    , c.AlgoDivision
    , c.TradeDivision
    , c.Items 
    , theme
    , seasonal_flag
)
, cte_aggregated_impressions AS ( 
SELECT 
   a.UniqueAdID
   , a.rundate
  , SUM(number_impressions) AS number_impressions
FROM 
  cte_all_ads AS a
  INNER JOIN IDENTIFIER(:catalog_schema_prefix || '.pctr_training_clicks_lookback') AS i
    ON i.control_sheet_AdID= a.UniqueAdID
    AND i.date BETWEEN a.rundate - (INTERVAL '1 DAY' * (:lookback_period + 1)) AND a.rundate- INTERVAL '1' DAY
  GROUP BY 
    a.UniqueAdID
   , a.rundate
)
, cte_total_impressions AS (
  SELECT 
  c.rundate 
  ,COALESCE(SUM(c.number_impressions),0) AS total_impressions 
FROM 
  cte_aggregated_impressions AS c
GROUP BY 
  c.rundate 
)
, cte_cumulative_impressions AS (
SELECT 
   a.UniqueAdID
   , a.rundate
   , COALESCE(a.number_impressions,0) AS number_impressions
   , COALESCE(a.number_impressions,0)/ t.total_impressions AS percentage_impressions
   , SUM(COALESCE(a.number_impressions,0))OVER (PARTITION BY a.rundate ORDER BY a.number_impressions DESC ) /t.total_impressions AS cumulative_percentage_impressions
FROM 
  cte_aggregated_impressions AS a
  INNER JOIN cte_total_impressions AS t
    ON t.rundate = a.rundate
)
-- Filter to ONLY include adverts above the impressions threshold 
SELECT 
  a.* 
  , COALESCE(c.number_impressions,0) AS number_impressions
  , COALESCE(c.percentage_impressions,0) AS percentage_impressions
  , COALESCE(c.cumulative_percentage_impressions,0) AS cumulative_percentage_impressions
FROM 
  cte_all_ads AS a
  LEFT JOIN cte_cumulative_impressions AS c
    ON c.UniqueAdID = a.UniqueAdID
    AND c.rundate = a.rundate
);
  

In [0]:
%sql
-- Customer Base:
--All customers who have bought in 365 days OR viewed in 60 days 

CREATE OR REPLACE TABLE IDENTIFIER(:catalog_schema_prefix || '.pctr_training_sessions_base') AS (
WITH cte_all_accounts AS (
SELECT 
    d.rundate
    ,account_number 
FROM 
    IDENTIFIER(:catalog_schema_prefix || '.pctr_build_year_baskets') AS b
    INNER JOIN IDENTIFIER(:catalog_schema_prefix || '.pctr_training_dates') AS d 
        ON b.order_date >= d.rundate - interval '365 days'
GROUP BY 
    d.rundate
    ,account_number 
UNION 
SELECT 
    d.rundate
    ,account_number
FROM 
    IDENTIFIER(:catalog_schema_prefix|| '.pctr_build_page_views') AS v 
    INNER JOIN IDENTIFIER(:catalog_schema_prefix || '.pctr_training_dates') AS d 
    ON v.viewdate >= d.rundate - interval '60 days'
GROUP BY 
    d.rundate
    ,account_number
)
SELECT 
     d.rundate
    , ac.account_number
    , NULL AS AdvertID
    , a.potnumber AS pot
    , a.campaignnumber AS campaign
    , a.versionnumber
    , c.accountstartdate
    , c.age 
    , c.gender
    , CASE WHEN c.mailoptout ='N' THEN 0 ELSE 1 END as mail_optout
    , c.postcodearea
    , CASE WHEN c.specialaccountindicator ='S' THEN 1 ELSE 0 END AS staff_indicator
    , CASE WHEN c.cashindicator= 'C' THEN 1 ELSE 0 END AS cash_acc
    , CAST(NULL AS timestamp) AS  ImpressionTimestamp
    , CAST(NULL AS timestamp) AS ClickTimestamp
    , CAST(NULL AS int) AS Ad_clicked 
    , a.UniqueAdID AS control_sheet_AdID
    , CAST(NULL AS string) AS treatment_type
    , CAST(NULL AS int) AS location
FROM 
    IDENTIFIER(:catalog_schema_prefix || '.pctr_training_dates') AS d
    INNER JOIN cte_all_accounts AS ac
        ON  ac.rundate =d.rundate 
    INNER JOIN marketingdata_prod.warehouse.svoccust AS c
        ON c.account_number=ac.account_number
        AND c.countrycode='GB'
        AND c.client='NEXT'
    --Join to adverts 
    INNER JOIN IDENTIFIER(:catalog_schema_prefix || '.pctr_training_ads_base') AS a
        ON  a.rundate=d.rundate
);


In [0]:
%sql
SELECT count(*) FROM IDENTIFIER(:catalog_schema_prefix || '.pctr_training_sessions_base')

In [0]:
%sql
    
ALTER TABLE IDENTIFIER(:catalog_schema_prefix || '.pctr_training_sessions_base')
ALTER COLUMN account_number SET NOT NULL;
ALTER TABLE IDENTIFIER(:catalog_schema_prefix || '.pctr_training_sessions_base')
ALTER COLUMN control_sheet_AdID SET NOT NULL;
ALTER TABLE IDENTIFIER(:catalog_schema_prefix || '.pctr_training_sessions_base')
ALTER COLUMN rundate SET NOT NULL ;
ALTER TABLE IDENTIFIER(:catalog_schema_prefix || '.pctr_training_sessions_base')
ADD PRIMARY KEY (account_number , control_sheet_AdID, rundate);

In [0]:
%sql
CREATE OR REPLACE TABLE IDENTIFIER(:catalog_schema_prefix || '.pctr_training_customer_base')  
AS ( 
SELECT 
     rundate 
    , account_number 
FROM 
   IDENTIFIER(:catalog_schema_prefix || '.pctr_training_sessions_base') 
GROUP BY rundate, account_number
);  